# 02 OSM and fire data

This notebook loads OSM context and FIRMS wildfire hotspots for the demo bounding box.

In [ ]:
from src.config import bounding_box
from src.fire_data import build_cell_grid, build_fire_dataset, load_firms_hotspots
from src.osm import fetch_bounding_box_features, feature_summary
from src.features import build_context_map

bbox = bounding_box()
cell_grid = build_cell_grid(bounding_box=bbox)

print(f"Grid: {cell_grid.n_rows} rows x {cell_grid.n_cols} cols")
print(f"Cell spacing: {cell_grid.spacing_deg:.4f} deg")

# OSM
try:
    snapshot = fetch_bounding_box_features(bounding_box=bbox)
    print(feature_summary(snapshot))
    context = build_context_map(snapshot, cell_grid=cell_grid)
    print("Water cells:", int(context.is_water.sum()))
    print("Settlement cells:", int(context.is_settlement.sum()))
    print("Peat-like cells:", int(context.is_peat_risk.sum()))
except Exception as e:
    print("OSM fetch skipped:", e)

# FIRMS
try:
    df = load_firms_hotspots()
    dataset = build_fire_dataset(df, bounding_box=bbox, cell_grid=cell_grid)
    print("Hotspots in box:", len(dataset.hotspots))
    print("Labeled days:", len(dataset.labeled_days))
    print("Labeled cells:", len(dataset.labels))
except Exception as e:
    print("FIRMS load skipped:", e)